## Transform Sprints Data
- Read bronze Sprints table
- Keep only the columns required for analytics(Drop url column)
- Standardise column name using snakecase(constructorId->constructor_id,driverId->driver_id)
- Rename columns to make them more meaningful(grid->grid_position)
- Filter out rows where seaon,round,constructor_id and driver_id is null (business key validation)
- Remove duplicate records.
- Transform values of column race_name to Title Case
- Write the transformed data to silver Sprints table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/03.silver-help

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.sprints"
silver_table=f"{catalog_name}.{silver_schema}.sprints"

In [0]:
import pyspark.sql.functions as F

### Step 1,2,3&4 Read bronze results table,select only required colum and standardise column names

In [0]:
sprints_df=(
    spark.table(bronze_table)
    .drop("url")
    .withColumnsRenamed(
        {"constructorId":"constructor_id",
         "driverId":"driver_id",
         "raceName":"race_name",
         "date":"race_date",
         "time":"race_time",
         "grid":"grid_position",
         "laps":"completed_laps",
         "number":"car_number",
         "position":"final_position",
         "positionText":"final_position_text"
        }
    )
    .filter(F.col("batch_id") == v_batch_id)
)
    


### Step 5 & 6  Apply Data Quality Checks
- Filter out rows where seaon,round,constructor_id and driver_id is null (business key validation)
- Remove Duplicates

In [0]:
sprints_valid_df=(
    sprints_df.filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("driver_id").isNotNull() &
        F.col("constructor_id").isNotNull()
    )
    .dropDuplicates(["season","round","driver_id","constructor_id"])
)



In [0]:

display(sprints_valid_df)



### Step -7 Transform values of column race_name to Title Case

In [0]:
sprints_final_df=(
    sprints_valid_df
    .withColumn("race_name",F.initcap(F.col("race_name")))
)
#display(drivers_final_df)

### Step-8 Write the transformed data to silver drivers table

In [0]:
write_to_silver(
    sprints_final_df,
    silver_table,
    merge_condition="s.season=t.season AND s.round=t.round AND s.constructor_id=t.constructor_id AND s.driver_id =t.driver_id",
    columns_to_update=[
       "race_date",
       "race_name",
        "round",
        "season",
       "constructor_id",
       "driver_id",
       "grid_position",
      "completed_laps",
       "car_number",
       "points",
       "final_position",
       "final_position_text",
       "status",
       "ingestion_timestamp",
        "source_file",
        "batch_id" 
    ]
)

In [0]:
display(spark.read.table(silver_table))